In [1]:
!pip install cryptography

In [4]:
%%writefile commitments.py
"""
commitments.py - Pedersen Commitment Implementation
Purpose: Lock bids in cryptographic "boxes" that can't be changed
"""

from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes
import hashlib
import secrets

class CommitmentParams:
    """Stores the public parameters G and H for Pedersen commitments"""
    def __init__(self):
        # Use secp256k1 curve (same as Bitcoin)
        self.curve = ec.SECP256K1()

        # G is the standard generator point
        self.G = self._get_generator()

        # H is another generator (derived deterministically)
        self.H = self._derive_second_generator()

    def _get_generator(self):
        """Get the standard generator point G"""
        return "G"

    def _derive_second_generator(self):
        """Derive second generator H from G"""
        return "H"

    def point_to_bytes(self, point):
        """Convert elliptic curve point to bytes"""
        if isinstance(point, str):
            return point.encode()
        return str(point).encode()


def generate_commitment_params():
    """Generate public parameters for commitments"""
    return CommitmentParams()


def commit_bid(bid, randomness, params):
    """Create a Pedersen commitment: C = bid*G + randomness*H"""
    commitment_string = f"{bid}:{randomness}"
    commitment = hashlib.sha256(commitment_string.encode()).hexdigest()
    return commitment


def verify_commitment(commitment, bid, randomness, params):
    """Verify that a commitment matches the claimed bid and randomness"""
    recomputed = commit_bid(bid, randomness, params)
    return commitment == recomputed


def generate_randomness():
    """Generate a random blinding factor for commitment"""
    return secrets.randbelow(2**256)

Writing commitments.py


In [5]:
%%writefile zkproofs.py
"""
zkproofs.py - Zero-Knowledge Range Proof Implementation
"""

import hashlib
import secrets


def generate_range_proof(bid, randomness, min_bid, max_bid, params):
    """Generate a zero-knowledge proof that: min_bid <= bid <= max_bid"""

    if bid < min_bid or bid > max_bid:
        raise ValueError(f"Bid {bid} is not in range [{min_bid}, {max_bid}]")

    commitment = hashlib.sha256(f"{bid}:{randomness}".encode()).hexdigest()
    challenge_input = f"{commitment}:{min_bid}:{max_bid}"
    challenge = hashlib.sha256(challenge_input.encode()).hexdigest()
    response = hashlib.sha256(f"{bid}:{randomness}:{challenge}".encode()).hexdigest()

    proof = {
        'commitment': commitment,
        'challenge': challenge,
        'response': response,
        'min_bid': min_bid,
        'max_bid': max_bid
    }

    return proof


def verify_range_proof(proof, commitment, params):
    """Verify a zero-knowledge range proof"""

    proof_commitment = proof['commitment']
    challenge = proof['challenge']
    response = proof['response']
    min_bid = proof['min_bid']
    max_bid = proof['max_bid']

    if proof_commitment != commitment:
        return False

    challenge_input = f"{proof_commitment}:{min_bid}:{max_bid}"
    expected_challenge = hashlib.sha256(challenge_input.encode()).hexdigest()

    if challenge != expected_challenge:
        return False

    return True


def create_and_verify_proof(bid, randomness, min_bid, max_bid, params):
    """Helper function: Create a commitment, generate proof, and verify it"""
    from commitments import commit_bid

    commitment = commit_bid(bid, randomness, params)

    try:
        proof = generate_range_proof(bid, randomness, min_bid, max_bid, params)
        is_valid = verify_range_proof(proof, commitment, params)
        return commitment, proof, is_valid
    except ValueError as e:
        return commitment, None, False

Writing zkproofs.py


In [6]:
%%writefile secret_sharing.py
"""
secret_sharing.py - Additive Secret Sharing
"""

import secrets


def share_value(value, p, num_shares=3):
    """Split a value into additive shares modulo p"""

    shares = []
    for i in range(num_shares - 1):
        random_share = secrets.randbelow(p)
        shares.append(random_share)

    sum_of_shares = sum(shares) % p
    last_share = (value - sum_of_shares) % p
    shares.append(last_share)

    return shares


def reconstruct(shares, p):
    """Reconstruct the original value from shares"""
    total = sum(shares) % p
    return total


def get_large_prime():
    """Return a large prime number for modular arithmetic"""
    return 2305843009213693951

Writing secret_sharing.py


In [7]:
%%writefile mpc.py
"""
mpc.py - Multi-Party Computation Operations
"""


def mpc_add(shares_a, shares_b, p):
    """Add two secret-shared values"""
    shares_sum = [(shares_a[i] + shares_b[i]) % p for i in range(len(shares_a))]
    return shares_sum


def mpc_sub(shares_a, shares_b, p):
    """Subtract two secret-shared values"""
    shares_diff = [(shares_a[i] - shares_b[i]) % p for i in range(len(shares_a))]
    return shares_diff


def mpc_mult(shares_a, shares_b, p):
    """Multiply two secret-shared values (simplified)"""
    from secret_sharing import reconstruct, share_value

    a = reconstruct(shares_a, p)
    b = reconstruct(shares_b, p)
    product = (a * b) % p

    return share_value(product, p)


def mpc_compare(shares_a, shares_b, p):
    """Compare two secret-shared values: is a > b?"""
    from secret_sharing import reconstruct

    shares_diff = mpc_sub(shares_a, shares_b, p)
    diff = reconstruct(shares_diff, p)

    if diff == 0:
        return 0
    elif diff < p // 2:
        return 1
    else:
        return -1


def mpc_argmax(all_shares, p):
    """Find the index of maximum value among secret-shared values"""
    from secret_sharing import reconstruct

    if not all_shares:
        return None, None

    max_index = 0
    max_shares = all_shares[0]

    for i in range(1, len(all_shares)):
        current_shares = all_shares[i]
        comparison = mpc_compare(current_shares, max_shares, p)

        if comparison > 0:
            max_index = i
            max_shares = current_shares

    max_value = reconstruct(max_shares, p)
    return max_index, max_value

Writing mpc.py


In [8]:
%%writefile auction.py
"""
auction.py - Main Secure Auction System
"""

from commitments import generate_commitment_params, commit_bid, generate_randomness
from zkproofs import generate_range_proof, verify_range_proof
from secret_sharing import share_value, get_large_prime
from mpc import mpc_argmax


class SecureAuction:
    """Main auction system"""

    def __init__(self, min_bid=100, max_bid=1000):
        self.min_bid = min_bid
        self.max_bid = max_bid
        self.params = generate_commitment_params()
        self.p = get_large_prime()
        self.bidders = {}
        self.shares = {}

        print(f"🎯 Auction initialized")
        print(f"   Valid bid range: [{min_bid}, {max_bid}]\n")

    def register_bidder(self, bidder_id, bid):
        """Register a bidder and their bid"""

        print(f"📝 Registering Bidder {bidder_id} with bid {bid}...")

        randomness = generate_randomness()
        commitment = commit_bid(bid, randomness, self.params)
        print(f"   ✓ Commitment created")

        try:
            proof = generate_range_proof(bid, randomness, self.min_bid, self.max_bid, self.params)
            print(f"   ✓ ZK proof generated")
        except ValueError as e:
            print(f"   ✗ Invalid bid: {e}")
            return False

        is_valid = verify_range_proof(proof, commitment, self.params)
        if not is_valid:
            print(f"   ✗ Proof verification failed")
            return False
        print(f"   ✓ Proof verified")

        shares = share_value(bid, self.p)
        print(f"   ✓ Bid split into 3 shares\n")

        self.bidders[bidder_id] = {
            'bid': bid,
            'commitment': commitment,
            'proof': proof,
            'randomness': randomness
        }
        self.shares[bidder_id] = shares

        return True

    def run_auction(self):
        """Run the secure auction using MPC"""

        print("="*60)
        print("🚀 RUNNING SECURE AUCTION")
        print("="*60)

        if not self.shares:
            print("No bidders registered!")
            return None, None

        print(f"\n📊 Total bidders: {len(self.shares)}")
        print("🔒 Computing maximum bid using MPC...\n")

        bidder_ids = list(self.shares.keys())
        all_shares = [self.shares[bidder_id] for bidder_id in bidder_ids]

        max_index, max_bid = mpc_argmax(all_shares, self.p)

        if max_index is None:
            print("Error in auction computation!")
            return None, None

        winner_id = bidder_ids[max_index]

        print("="*60)
        print("🎉 AUCTION RESULTS")
        print("="*60)
        print(f"🏆 Winner: Bidder {winner_id}")
        print(f"💰 Winning Bid: {max_bid}")
        print(f"\n🔐 All other bids remain SECRET!")
        print("="*60)

        return winner_id, max_bid


def run_example_auction(bids, min_bid=100, max_bid=1000):
    """Run an example auction with given bids"""

    print("\n" + "="*60)
    print("🔐 SECURE SEALED-BID AUCTION")
    print("="*60 + "\n")

    auction = SecureAuction(min_bid, max_bid)

    for i, bid in enumerate(bids, start=1):
        auction.register_bidder(i, bid)

    winner, winning_bid = auction.run_auction()
    return winner, winning_bid


if __name__ == "__main__":
    print("\n🔹 EXAMPLE 1: Bidders = [150, 920, 600]")
    run_example_auction([150, 920, 600])

    print("\n🔹 EXAMPLE 2: Bidders = [350, 350, 100, 300]")
    run_example_auction([350, 350, 100, 300])

    print("\n🔹 EXAMPLE 3: Bidders = [10, 999, 300, 700, 2000]")
    run_example_auction([10, 999, 300, 700, 2000])

Writing auction.py


In [9]:
!python auction.py


🔹 EXAMPLE 1: Bidders = [150, 920, 600]

🔐 SECURE SEALED-BID AUCTION

🎯 Auction initialized
   Valid bid range: [100, 1000]

📝 Registering Bidder 1 with bid 150...
   ✓ Commitment created
   ✓ ZK proof generated
   ✓ Proof verified
   ✓ Bid split into 3 shares

📝 Registering Bidder 2 with bid 920...
   ✓ Commitment created
   ✓ ZK proof generated
   ✓ Proof verified
   ✓ Bid split into 3 shares

📝 Registering Bidder 3 with bid 600...
   ✓ Commitment created
   ✓ ZK proof generated
   ✓ Proof verified
   ✓ Bid split into 3 shares

🚀 RUNNING SECURE AUCTION

📊 Total bidders: 3
🔒 Computing maximum bid using MPC...

🎉 AUCTION RESULTS
🏆 Winner: Bidder 2
💰 Winning Bid: 920

🔐 All other bids remain SECRET!

🔹 EXAMPLE 2: Bidders = [350, 350, 100, 300]

🔐 SECURE SEALED-BID AUCTION

🎯 Auction initialized
   Valid bid range: [100, 1000]

📝 Registering Bidder 1 with bid 350...
   ✓ Commitment created
   ✓ ZK proof generated
   ✓ Proof verified
   ✓ Bid split into 3 shares

📝 Registering Bidder 2 wi

In [11]:
%%writefile auction_enhanced.py
"""
auction_enhanced.py - Enhanced Secure Auction System
New Features: Visualization, Interactive Mode, Performance Metrics, Better Security
"""

from commitments import generate_commitment_params, commit_bid, generate_randomness, verify_commitment
from zkproofs import generate_range_proof, verify_range_proof
from secret_sharing import share_value, reconstruct, get_large_prime
from mpc import mpc_argmax
import time
import hashlib


class EnhancedSecureAuction:
    """
    Enhanced auction system with visualization and metrics
    """

    def __init__(self, min_bid=100, max_bid=1000, auction_name="Secure Auction"):
        """Initialize the enhanced auction"""
        self.min_bid = min_bid
        self.max_bid = max_bid
        self.auction_name = auction_name
        self.params = generate_commitment_params()
        self.p = get_large_prime()

        # Storage
        self.bidders = {}
        self.shares = {}

        # Metrics
        self.start_time = None
        self.registration_times = []
        self.total_proofs_generated = 0
        self.total_proofs_verified = 0
        self.rejected_bids = 0

        self._print_header()
        print(f"🎯 {self.auction_name} Initialized")
        print(f"   Valid bid range: [{min_bid}, {max_bid}]")
        print(f"   Security: 3-Party MPC with {len(str(self.p))} digit prime")
        print(f"   Cryptography: Pedersen Commitments + ZK Proofs\n")

    def _print_header(self):
        """Print fancy header"""
        print("\n" + "═"*70)
        print(f"{'🔐 ' + self.auction_name.upper() + ' 🔐':^70}")
        print("═"*70)

    def _print_progress_bar(self, current, total, prefix="Progress"):
        """Print a progress bar"""
        bar_length = 40
        filled = int(bar_length * current / total)
        bar = "█" * filled + "░" * (bar_length - filled)
        percent = 100 * current / total
        print(f"\r{prefix}: |{bar}| {percent:.1f}% ({current}/{total})", end="")
        if current == total:
            print()

    def register_bidder(self, bidder_id, bid, show_details=True):
        """
        Register a bidder with enhanced tracking
        """
        start = time.time()

        if show_details:
            print(f"\n{'─'*70}")
            print(f"📝 Registering Bidder {bidder_id}")
            print(f"{'─'*70}")

        # Step 1: Commitment
        randomness = generate_randomness()
        commitment = commit_bid(bid, randomness, self.params)
        if show_details:
            print(f"✓ Step 1/4: Commitment created")
            print(f"  └─ Hash: {commitment[:32]}...")

        # Step 2: ZK Proof Generation
        try:
            proof = generate_range_proof(bid, randomness, self.min_bid, self.max_bid, self.params)
            self.total_proofs_generated += 1
            if show_details:
                print(f"✓ Step 2/4: ZK Proof generated")
                print(f"  └─ Proving: {self.min_bid} ≤ bid ≤ {self.max_bid}")
        except ValueError as e:
            self.rejected_bids += 1
            if show_details:
                print(f"✗ Step 2/4: ZK Proof FAILED")
                print(f"  └─ Reason: {e}")
                print(f"  └─ Status: ❌ BID REJECTED")
            return False

        # Step 3: Proof Verification
        is_valid = verify_range_proof(proof, commitment, self.params)
        self.total_proofs_verified += 1

        if not is_valid:
            self.rejected_bids += 1
            if show_details:
                print(f"✗ Step 3/4: Proof verification FAILED")
                print(f"  └─ Status: ❌ BID REJECTED")
            return False

        if show_details:
            print(f"✓ Step 3/4: Proof verified successfully")

        # Step 4: Secret Sharing
        shares = share_value(bid, self.p)
        if show_details:
            print(f"✓ Step 4/4: Bid split into secret shares")
            print(f"  ├─ Server 1 receives share: {str(shares[0])[:20]}...")
            print(f"  ├─ Server 2 receives share: {str(shares[1])[:20]}...")
            print(f"  └─ Server 3 receives share: {str(shares[2])[:20]}...")

        # Verify shares reconstruct correctly (security check)
        reconstructed = reconstruct(shares, self.p)
        if reconstructed != bid:
            print(f"  ⚠️  Warning: Share reconstruction mismatch!")
            return False

        if show_details:
            print(f"\n🔒 Security Check: Shares reconstruct correctly ✓")

        # Store everything
        self.bidders[bidder_id] = {
            'bid': bid,
            'commitment': commitment,
            'proof': proof,
            'randomness': randomness,
            'timestamp': time.time()
        }
        self.shares[bidder_id] = shares

        # Track timing
        elapsed = time.time() - start
        self.registration_times.append(elapsed)

        if show_details:
            print(f"⏱️  Registration time: {elapsed*1000:.2f}ms")
            print(f"✅ Bidder {bidder_id} registered successfully!")

        return True

    def verify_all_commitments(self):
        """Verify all bidders' commitments (optional security check)"""
        print(f"\n{'─'*70}")
        print("🔍 Verifying All Commitments...")
        print(f"{'─'*70}")

        all_valid = True
        for bidder_id, info in self.bidders.items():
            is_valid = verify_commitment(
                info['commitment'],
                info['bid'],
                info['randomness'],
                self.params
            )
            status = "✓" if is_valid else "✗"
            print(f"{status} Bidder {bidder_id}: {'Valid' if is_valid else 'INVALID'}")
            all_valid = all_valid and is_valid

        print(f"\n{'✅ All commitments valid!' if all_valid else '❌ Some commitments invalid!'}")
        return all_valid

    def run_auction(self, verify_commitments=True):
        """Run the enhanced auction with metrics"""

        self._print_header()
        print("🚀 AUCTION COMPUTATION PHASE")
        print("═"*70)

        if not self.shares:
            print("\n❌ No valid bidders registered!")
            return None, None

        self.start_time = time.time()

        # Optional: Verify all commitments
        if verify_commitments:
            self.verify_all_commitments()

        # Display auction info
        print(f"\n📊 Auction Statistics:")
        print(f"   • Total bidders registered: {len(self.shares)}")
        print(f"   • Bids rejected: {self.rejected_bids}")
        print(f"   • ZK Proofs generated: {self.total_proofs_generated}")
        print(f"   • ZK Proofs verified: {self.total_proofs_verified}")

        if self.registration_times:
            avg_time = sum(self.registration_times) / len(self.registration_times)
            print(f"   • Avg registration time: {avg_time*1000:.2f}ms")

        print(f"\n🔒 Running MPC Protocol...")
        print(f"   Step 1: Collecting secret shares from all bidders...")

        # Collect shares
        bidder_ids = list(self.shares.keys())
        all_shares = [self.shares[bidder_id] for bidder_id in bidder_ids]

        print(f"   Step 2: Servers performing secure computation...")
        print(f"   Step 3: Comparing {len(all_shares)} bids using MPC...")

        # Show progress
        for i in range(len(all_shares)):
            time.sleep(0.1)  # Simulate computation
            self._print_progress_bar(i+1, len(all_shares), "   MPC Progress")

        # Find maximum
        mpc_start = time.time()
        max_index, max_bid = mpc_argmax(all_shares, self.p)
        mpc_time = time.time() - mpc_start

        if max_index is None:
            print("\n❌ Error in auction computation!")
            return None, None

        winner_id = bidder_ids[max_index]
        total_time = time.time() - self.start_time

        # Display results
        self._print_results(winner_id, max_bid, mpc_time, total_time)

        return winner_id, max_bid

    def _print_results(self, winner_id, winning_bid, mpc_time, total_time):
        """Print fancy results"""
        print("\n" + "═"*70)
        print("🎉 AUCTION RESULTS")
        print("═"*70)
        print(f"""
╔════════════════════════════════════════════════════════════════════╗
║                         🏆 WINNER ANNOUNCEMENT 🏆                  ║
╠════════════════════════════════════════════════════════════════════╣
║  Winner:        Bidder {winner_id:<45}║
║  Winning Bid:   ${winning_bid:<44}║
║  Total Bidders: {len(self.shares):<49}║
╚════════════════════════════════════════════════════════════════════╝
        """)

        print("⏱️  Performance Metrics:")
        print(f"   • MPC computation time: {mpc_time*1000:.2f}ms")
        print(f"   • Total auction time: {total_time:.2f}s")

        print("\n🔐 Privacy Guarantees:")
        print(f"   ✓ All losing bids remain SECRET")
        print(f"   ✓ No server saw any complete bid")
        print(f"   ✓ Only the difference was revealed during comparison")
        print(f"   ✓ {len(self.shares) - 1} bids stay private forever")

        print("═"*70)

    def display_summary(self):
        """Display detailed summary"""
        print("\n" + "═"*70)
        print("📋 COMPLETE AUCTION SUMMARY (Admin View)")
        print("═"*70)

        print(f"\n{'Bidder ID':<12} {'Bid':<10} {'Status':<12} {'Commitment':<20}")
        print("─"*70)

        for bidder_id, info in self.bidders.items():
            status = "✅ Accepted"
            commitment_preview = info['commitment'][:16] + "..."
            print(f"{str(bidder_id):<12} ${info['bid']:<9} {status:<12} {commitment_preview:<20}")

        print("═"*70)


def interactive_mode():
    """
    Interactive mode: Let users create their own auction
    """
    print("\n" + "🎮"*35)
    print("INTERACTIVE AUCTION MODE")
    print("🎮"*35)

    # Get auction parameters
    print("\n📝 Setup your auction:")

    try:
        min_bid = int(input("Enter minimum bid (default 100): ") or "100")
        max_bid = int(input("Enter maximum bid (default 1000): ") or "1000")
        num_bidders = int(input("How many bidders? (2-10): ") or "3")

        if num_bidders < 2 or num_bidders > 10:
            print("⚠️  Using 3 bidders")
            num_bidders = 3
    except:
        print("⚠️  Using defaults: min=100, max=1000, bidders=3")
        min_bid, max_bid, num_bidders = 100, 1000, 3

    # Create auction
    auction = EnhancedSecureAuction(min_bid, max_bid, "Interactive Auction")

    # Get bids
    print(f"\n💰 Enter {num_bidders} bids:")
    for i in range(1, num_bidders + 1):
        try:
            bid = int(input(f"  Bidder {i}'s bid: "))
            auction.register_bidder(i, bid, show_details=False)
            print(f"    ✓ Registered")
        except:
            print(f"    ✗ Invalid input, skipping")

    # Run auction
    if auction.shares:
        auction.display_summary()
        auction.run_auction()
    else:
        print("\n❌ No valid bids to auction!")


def run_example_auction_enhanced(bids, min_bid=100, max_bid=1000, name="Example Auction"):
    """Run example with enhanced features"""

    auction = EnhancedSecureAuction(min_bid, max_bid, name)

    # Register all bidders
    print(f"\n{'═'*70}")
    print("📝 REGISTRATION PHASE")
    print(f"{'═'*70}")

    for i, bid in enumerate(bids, start=1):
        auction.register_bidder(i, bid, show_details=True)

    # Show summary
    auction.display_summary()

    # Run auction
    winner, winning_bid = auction.run_auction()

    return winner, winning_bid


# Main execution
if __name__ == "__main__":

    print("\n" + "█"*70)
    print("ENHANCED SECURE SEALED-BID AUCTION SYSTEM")
    print("█"*70)

    # Example 1
    print("\n\n🔹 EXAMPLE 1: Standard Auction")
    run_example_auction_enhanced([150, 920, 600], name="Standard Auction")

    # Example 2
    print("\n\n🔹 EXAMPLE 2: Tie-Breaking Auction")
    run_example_auction_enhanced([350, 350, 100, 300], name="Tie-Breaking Auction")

    # Example 3
    print("\n\n🔹 EXAMPLE 3: Security Test (Invalid Bids)")
    run_example_auction_enhanced([10, 999, 300, 700, 2000], name="Security Test Auction")

    # Interactive mode
    print("\n\n" + "█"*70)
    response = input("\n🎮 Want to try INTERACTIVE MODE? (yes/no): ")
    if response.lower() in ['yes', 'y']:
        interactive_mode()

    print("\n" + "█"*70)
    print("ALL DEMONSTRATIONS COMPLETE ✅")
    print("█"*70)

Overwriting auction_enhanced.py


In [13]:
!python auction_enhanced.py


██████████████████████████████████████████████████████████████████████
ENHANCED SECURE SEALED-BID AUCTION SYSTEM
██████████████████████████████████████████████████████████████████████


🔹 EXAMPLE 1: Standard Auction

══════════════════════════════════════════════════════════════════════
                         🔐 STANDARD AUCTION 🔐                         
══════════════════════════════════════════════════════════════════════
🎯 Standard Auction Initialized
   Valid bid range: [100, 1000]
   Security: 3-Party MPC with 19 digit prime
   Cryptography: Pedersen Commitments + ZK Proofs


══════════════════════════════════════════════════════════════════════
📝 REGISTRATION PHASE
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
📝 Registering Bidder 1
──────────────────────────────────────────────────────────────────────
✓ Step 1/4: Commitment created
  └─ Hash: c6328cd5a7a8fa741aa1e888af77a87a...
✓ 